<a href="https://colab.research.google.com/github/VasilisPapageorgiou/Amortization-of-Risk-Indicators/blob/main/Amortization_Exp5_3_B.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# =====================================================================================
# EXPERIMENT 5.3-B — COVERAGE vs LABEL PRECISION
# FINAL / JASA-LEVEL / RESUMABLE GOOGLE-DRIVE VERSION
#
# Scientific question
# -------------------
# Under a fixed Monte Carlo training-label budget R*m = 50,000,
# should simulation effort be allocated to:
#
#   (i)  more epidemic configurations R  -> broader coverage
#   (ii) more simulations/configuration m -> more precise labels?
#
# Fixed-budget allocations:
#
#   (R,m) =
#       (200,250),
#       (500,100),
#       (1000,50),
#       (2000,25),
#       (5000,10)
#
# Main improvements relative to the preliminary version
# ------------------------------------------------------
# 1. 5 independent MC/training replications
# 2. nested configurations and nested trajectories
# 3. balanced nested configuration ordering
# 4. exact reference repeated under the same optimization seeds
# 5. common exact validation set for controlled model selection
# 6. seen-N and held-out-N test sets
# 7. direct MC label-noise diagnostics
# 8. hierarchical paired bootstrap 95% confidence intervals
# 9. persistent Google-Drive checkpoints
# 10. RESUME / START-NEW protection
#
# Main manuscript outputs
# -----------------------
# 1. main_table_5_3B.tex
# 2. figure_5_3B_main.pdf
#
# Additional analysis outputs
# ---------------------------
# - raw_test_errors.csv
# - teacher_label_errors.csv
# - robustness_by_stratum.csv
# - extreme_allocation_contrast.csv
#
# CPU ONLY
# SPARSE LU ONLY
# NO MATRIX INVERSE
# =====================================================================================


# =====================================================================================
# 0. GOOGLE DRIVE
# =====================================================================================

from google.colab import drive

drive.mount(
    "/content/drive",
    force_remount=False
)


# =====================================================================================
# 1. IMPORTS
# =====================================================================================

import os

os.environ["OMP_NUM_THREADS"] = "1"
os.environ["OPENBLAS_NUM_THREADS"] = "1"
os.environ["MKL_NUM_THREADS"] = "1"
os.environ["NUMEXPR_NUM_THREADS"] = "1"

import json
import hashlib
import math
import pickle
import random
import time

from dataclasses import dataclass, asdict
from datetime import datetime
from functools import lru_cache
from pathlib import Path
from typing import Tuple

import numpy as np
import pandas as pd

from scipy import sparse
from scipy.sparse.linalg import splu
from scipy.stats import qmc

from joblib import Parallel, delayed

from numba import (
    njit,
    prange,
    set_num_threads,
    get_num_threads
)

import torch
import torch.nn as nn

import matplotlib.pyplot as plt


# =====================================================================================
# 2. CONFIGURATION
# =====================================================================================

@dataclass
class C:

    seed:int = 20260820

    beta:Tuple[float,float] = (.30,1.50)
    gamma:Tuple[float,float] = (.20,1.00)
    omega:Tuple[float,float] = (.02,.50)
    frac:Tuple[float,float] = (.02,.20)

    trainN:Tuple[int,...] = tuple(
        range(40,401,20)
    )

    width:int = 128
    depth:int = 3

    batch:int = 64

    # Common manuscript configuration
    epochs:int = 500

    lr:float = 1e-3
    wd:float = 1e-6

    patience:int = 20
    delta:float = 1e-6
    clip:float = 5.


cfg = C()


# =====================================================================================
# 3. SCIENTIFIC EXPERIMENT SETTINGS
# =====================================================================================

ALLOC = (
    (200,250),
    (500,100),
    (1000,50),
    (2000,25),
    (5000,10)
)

BUDGET = 50_000

assert all(
    R*m == BUDGET
    for R,m in ALLOC
)


RMAX = max(
    R
    for R,m in ALLOC
)


# -------------------------------------------------------------------------
# Independent repetitions of:
#   MC labels + neural optimization
# -------------------------------------------------------------------------

N_REP = 5


# -------------------------------------------------------------------------
# Validation
#
# 20 configurations per training-grid population size:
# 19 N values * 20 = 380
#
# The exact validation set is COMMON to all teacher mechanisms.
# It is used only for controlled early stopping/model selection.
# The fixed MC budget refers to MC TRAINING labels.
# -------------------------------------------------------------------------

VAL_PER_N = 20
N_VAL = (
    len(cfg.trainN)
    *
    VAL_PER_N
)


# -------------------------------------------------------------------------
# Test sets
#
# Seen:
#   N = 40,60,...,400
#
# Held-out interpolation:
#   N = 50,70,...,390
#
# Exactly 20 configurations per N.
# -------------------------------------------------------------------------

SEEN_N = cfg.trainN

INTERP_N = tuple(
    range(50,400,20)
)

TEST_PER_N = 20

N_TEST_SEEN = (
    len(SEEN_N)
    *
    TEST_PER_N
)

N_TEST_INTERP = (
    len(INTERP_N)
    *
    TEST_PER_N
)


assert set(
    SEEN_N
).isdisjoint(
    INTERP_N
)


Nscale = max(
    cfg.trainN
)


# -------------------------------------------------------------------------
# Uncertainty
# -------------------------------------------------------------------------

BOOTSTRAP_B = 3000


# -------------------------------------------------------------------------
# Checkpoint granularity
# -------------------------------------------------------------------------

EXACT_CHUNK = 100
MC_CHUNK = 250
CHECKPOINT_EVERY = 5


# =====================================================================================
# 4. SCIENTIFIC VERSION
#
# IMPORTANT:
# If you change the mathematical implementation of:
#   - exact_p
#   - sim_C
#   - loss
#   - architecture
#   - experimental design logic
#
# increment CODE_VERSION.
#
# Cosmetic figure changes do NOT require a new code version.
# =====================================================================================

CODE_VERSION = "5.3B_JASA_v1"


SCIENTIFIC_CONFIG = {

    "code_version":
        CODE_VERSION,

    "cfg":
        asdict(cfg),

    "ALLOC":
        ALLOC,

    "BUDGET":
        BUDGET,

    "N_REP":
        N_REP,

    "VAL_PER_N":
        VAL_PER_N,

    "TEST_PER_N":
        TEST_PER_N,

    "SEEN_N":
        SEEN_N,

    "INTERP_N":
        INTERP_N,

    "BOOTSTRAP_B":
        BOOTSTRAP_B,

    "validation_protocol":
        (
            "common exact validation set used only "
            "for controlled early stopping/model selection"
        ),

    "MC_protocol":
        (
            "nested configurations and nested trajectories "
            "within each replication"
        )
}


def scientific_signature(x):

    text = json.dumps(
        x,
        sort_keys=True,
        default=str
    )

    return hashlib.sha256(
        text.encode()
    ).hexdigest()[:16]


CURRENT_SIGNATURE = scientific_signature(
    SCIENTIFIC_CONFIG
)


# =====================================================================================
# 5. RESUME OR START NEW
# =====================================================================================

BASE_ROOT = Path(
    "/content/drive/MyDrive/"
    "StatisticalLearning/"
    "Experiment_5_3B_JASA"
)

BASE_ROOT.mkdir(
    parents=True,
    exist_ok=True
)


print("\n" + "="*90)
print("EXPERIMENT 5.3-B — START MODE")
print("="*90)

print(
    "1 = RESUME latest experiment"
)

print(
    "2 = START NEW experiment from zero"
)

print("="*90)


choice = input(
    "Choose 1 or 2: "
).strip()


if choice not in (
    "1",
    "2"
):

    raise RuntimeError(
        "Rerun the cell and choose 1 or 2."
    )


LATEST_FILE = (
    BASE_ROOT
    /
    "latest_run.txt"
)


# -------------------------------------------------------------------------
# NEW
# -------------------------------------------------------------------------

if choice == "2":

    timestamp = datetime.now().strftime(
        "%Y%m%d_%H%M%S"
    )

    ROOT = (
        BASE_ROOT
        /
        f"run_{timestamp}"
    )

    ROOT.mkdir(
        parents=True,
        exist_ok=False
    )


    manifest = {

        "signature":
            CURRENT_SIGNATURE,

        "scientific_config":
            SCIENTIFIC_CONFIG,

        "created":
            timestamp

    }


    with open(
        ROOT/"manifest.pkl",
        "wb"
    ) as f:

        pickle.dump(
            manifest,
            f,
            pickle.HIGHEST_PROTOCOL
        )


    LATEST_FILE.write_text(
        ROOT.name
    )


    print(
        "\nSTARTING NEW EXPERIMENT"
    )


# -------------------------------------------------------------------------
# RESUME
# -------------------------------------------------------------------------

else:

    if not LATEST_FILE.exists():

        raise RuntimeError(
            "No previous experiment exists. Choose START NEW."
        )


    run_name = (
        LATEST_FILE
        .read_text()
        .strip()
    )


    ROOT = (
        BASE_ROOT
        /
        run_name
    )


    manifest_file = (
        ROOT
        /
        "manifest.pkl"
    )


    if not manifest_file.exists():

        raise RuntimeError(
            "Previous experiment has no manifest. "
            "Resume is unsafe."
        )


    with open(
        manifest_file,
        "rb"
    ) as f:

        manifest = pickle.load(
            f
        )


    old_signature = (
        manifest[
            "signature"
        ]
    )


    if (
        old_signature
        !=
        CURRENT_SIGNATURE
    ):

        print("\n" + "!"*90)

        print(
            "SCIENTIFIC CONFIGURATION HAS CHANGED"
        )

        print("!"*90)

        print(
            "Saved signature :",
            old_signature
        )

        print(
            "Current signature:",
            CURRENT_SIGNATURE
        )


        raise RuntimeError(
            "\nRESUME ABORTED.\n"
            "Choose START NEW to prevent heterogeneous computations."
        )


    print(
        "\nRESUMING EXISTING EXPERIMENT"
    )


print(
    "ROOT:",
    ROOT
)

print(
    "Scientific signature:",
    CURRENT_SIGNATURE
)


# =====================================================================================
# 6. DIRECTORIES
# =====================================================================================

CACHE = ROOT/"cache"

EXACT_CACHE = CACHE/"exact"
MC_CACHE = CACHE/"mc"

MODEL_CACHE = CACHE/"models"
CHECKPOINT_CACHE = CACHE/"checkpoints"

OUT = ROOT/"results"


for d in (
    CACHE,
    EXACT_CACHE,
    MC_CACHE,
    MODEL_CACHE,
    CHECKPOINT_CACHE,
    OUT
):

    d.mkdir(
        parents=True,
        exist_ok=True
    )


# =====================================================================================
# 7. ATOMIC SAVES
# =====================================================================================

def atomic_pickle(
    obj,
    path
):

    path = Path(path)

    tmp = path.with_suffix(
        path.suffix+".tmp"
    )


    with open(
        tmp,
        "wb"
    ) as f:

        pickle.dump(
            obj,
            f,
            pickle.HIGHEST_PROTOCOL
        )


    os.replace(
        tmp,
        path
    )


def atomic_torch_save(
    obj,
    path
):

    path = Path(path)

    tmp = path.with_suffix(
        path.suffix+".tmp"
    )


    torch.save(
        obj,
        tmp
    )


    os.replace(
        tmp,
        path
    )


def safe_pickle_load(
    path,
    default=None
):

    try:

        with open(
            path,
            "rb"
        ) as f:

            return pickle.load(
                f
            )

    except Exception:

        return default


# =====================================================================================
# 8. CPU SETTINGS
# =====================================================================================

CPU = (
    os.cpu_count()
    or
    1
)


N_EXACT = max(
    1,
    min(
        2,
        CPU
    )
)


N_MC = max(
    1,
    min(
        CPU,
        get_num_threads()
    )
)


set_num_threads(
    N_MC
)


TORCH_THREADS = max(
    1,
    min(
        8,
        CPU
    )
)


torch.set_num_threads(
    TORCH_THREADS
)


try:

    torch.set_num_interop_threads(
        1
    )

except RuntimeError:

    pass


def seed_all(s):

    random.seed(
        s
    )

    np.random.seed(
        s
    )

    torch.manual_seed(
        s
    )


seed_all(
    cfg.seed
)


print("\n"+"="*95)

print(
    "COMPUTATIONAL CONFIGURATION"
)

print("="*95)

print(
    "CPU cores:",
    CPU
)

print(
    "Exact workers:",
    N_EXACT
)

print(
    "Numba threads:",
    N_MC
)

print(
    "PyTorch threads:",
    torch.get_num_threads()
)

print(
    "Replications:",
    N_REP
)

print(
    "MC budget/allocation:",
    f"{BUDGET:,}"
)

print(
    "Allocations:",
    ALLOC
)


# =====================================================================================
# 9. DATA RECORD
# =====================================================================================

@dataclass
class Rec:

    b:float
    g:float
    w:float

    N:int
    i0:int

    p:np.ndarray


# =====================================================================================
# 10. SIRS TOPOLOGY
# =====================================================================================

@lru_cache(None)
def topo(N):

    st = [

        (s,i)

        for i in range(
            1,
            N+1
        )

        for s in range(
            N-i+1
        )

    ]


    ix = {

        x:j

        for j,x
        in enumerate(st)

    }


    M = len(
        st
    )


    ir=[]; ic=[]; ib=[]
    rr=[]; rc=[]; rb=[]
    wr=[]; wc=[]; wb=[]


    db = np.zeros(M)
    dg = np.zeros(M)
    dw = np.zeros(M)
    qb = np.zeros(M)


    for j,(s,i) in enumerate(
        st
    ):

        r = (
            N-s-i
        )


        # infection
        if s:

            ir.append(
                j
            )

            ic.append(
                ix[
                    (s-1,i+1)
                ]
            )

            rate = (
                s*i/N
            )

            ib.append(
                rate
            )

            db[j] = (
                rate
            )


        # recovery
        dg[j] = (
            i
        )


        if i==1:

            qb[j] = (
                i
            )

        else:

            rr.append(
                j
            )

            rc.append(
                ix[
                    (s,i-1)
                ]
            )

            rb.append(
                i
            )


        # immunity loss
        if r:

            wr.append(
                j
            )

            wc.append(
                ix[
                    (s+1,i)
                ]
            )

            wb.append(
                r
            )

            dw[j] = (
                r
            )


    A = (
        lambda x,d=float:
        np.asarray(
            x,
            dtype=d
        )
    )


    return (

        ix,
        M,

        A(ir,int),
        A(ic,int),
        A(ib),

        A(rr,int),
        A(rc,int),
        A(rb),

        A(wr,int),
        A(wc,int),
        A(wb),

        db,
        dg,
        dw,
        qb

    )


# =====================================================================================
# 11. EXACT TRUNCATED-WITH-OVERFLOW DISTRIBUTION
# =====================================================================================

def exact_p(
    b,
    g,
    w,
    N,
    i0
):

    (
        ix,
        M,

        ir,
        ic,
        ib,

        rr,
        rc,
        rb,

        wr,
        wc,
        wb,

        db,
        dg,
        dw,
        qb

    ) = topo(
        N
    )


    rows = np.r_[

        ir,
        rr,
        wr,
        np.arange(M)

    ]


    cols = np.r_[

        ic,
        rc,
        wc,
        np.arange(M)

    ]


    T = sparse.coo_matrix(

        (

            np.r_[

                b*ib,
                g*rb,
                w*wb,

                -(
                    b*db
                    +
                    g*dg
                    +
                    w*dw
                )

            ],

            (
                rows,
                cols
            )

        ),

        shape=(
            M,
            M
        )

    ).tocsc()


    D1 = sparse.coo_matrix(

        (
            b*ib,
            (
                ir,
                ic
            )
        ),

        shape=(
            M,
            M
        )

    ).tocsc()


    A0 = (
        -(T-D1)
    ).tocsc()


    lu = splu(
        A0,
        permc_spec="COLAMD"
    )


    q = (
        g*qb
    )


    v = np.zeros(
        M
    )


    v[
        ix[
            (
                N-i0,
                i0
            )
        ]
    ] = 1.


    bvec = lu.solve(
        q
    )


    D1T = (
        D1.T.tocsr()
    )


    p = np.zeros(
        N+2
    )


    for k in range(
        N+1
    ):

        p[k] = (
            v@bvec
        )


        v = np.asarray(

            D1T

            @

            lu.solve(
                v,
                trans="T"
            )

        ).ravel()


    p[-1] = (
        v.sum()
    )


    p[
        np.abs(p)<1e-12
    ] = 0.


    p = np.maximum(
        p,
        0.
    )


    mass = (
        p.sum()
    )


    if (
        not np.isfinite(
            mass
        )
        or
        mass<=0
    ):

        raise RuntimeError(
            "Invalid exact probability vector."
        )


    p /= (
        mass
    )


    return p


# =====================================================================================
# 12. DESIGN
# =====================================================================================

def design(
    n,
    Ns,
    seed
):

    U = qmc.LatinHypercube(
        4,
        seed=seed
    ).random(
        n
    )


    scale = lambda x,a: (

        a[0]
        +
        (
            a[1]-a[0]
        )*x

    )


    b = scale(
        U[:,0],
        cfg.beta
    )


    g = scale(
        U[:,1],
        cfg.gamma
    )


    w = scale(
        U[:,2],
        cfg.omega
    )


    f = scale(
        U[:,3],
        cfg.frac
    )


    Nv = np.tile(

        np.asarray(
            Ns
        ),

        math.ceil(
            n/len(Ns)
        )

    )[:n]


    rng = np.random.default_rng(
        seed+99
    )


    rng.shuffle(
        Nv
    )


    i0 = np.asarray([

        int(
            np.clip(
                round(
                    f[j]*Nv[j]
                ),
                2,
                Nv[j]
            )
        )

        for j in range(n)

    ])


    for N in Ns:

        z = np.where(
            Nv==N
        )[0]


        if len(z):

            k = max(
                1,
                round(
                    .25*len(z)
                )
            )


            i0[
                rng.choice(
                    z,
                    k,
                    replace=False
                )
            ] = 1


    return [

        (
            float(b[j]),
            float(g[j]),
            float(w[j]),
            int(Nv[j]),
            int(i0[j])
        )

        for j in range(n)

    ]


# =====================================================================================
# 13. BALANCED NESTED TRAINING ORDER
#
# This prevents the coverage effect from being confounded with an unbalanced
# representation of population sizes or i0=1 cases in small-R prefixes.
# =====================================================================================

def balanced_order(
    records,
    seed
):

    rng = np.random.default_rng(
        seed
    )


    groups = {}


    for N in cfg.trainN:

        for flag in (
            0,
            1
        ):

            idx = np.asarray(

                [

                    j

                    for j,r
                    in enumerate(records)

                    if
                    r.N==N
                    and
                    int(r.i0==1)==flag

                ],

                dtype=int

            )


            rng.shuffle(
                idx
            )


            groups[
                (N,flag)
            ] = list(
                idx
            )


    ptr = {
        k:0
        for k in groups
    }


    usedN = {
        N:0
        for N in cfg.trainN
    }


    used1 = 0

    order = []


    for position in range(
        len(records)
    ):

        target1 = (
            .25
            *
            (
                position+1
            )
        )


        preferred_flag = (
            1
            if used1<target1
            else 0
        )


        chosen = None


        for flag in (
            preferred_flag,
            1-preferred_flag
        ):

            candidates = [

                N

                for N in cfg.trainN

                if
                ptr[
                    (N,flag)
                ]
                <
                len(
                    groups[
                        (N,flag)
                    ]
                )

            ]


            if candidates:

                m = min(
                    usedN[N]
                    for N in candidates
                )


                candidates = [

                    N

                    for N in candidates

                    if usedN[N]==m

                ]


                N = int(
                    rng.choice(
                        candidates
                    )
                )


                chosen = (
                    N,
                    flag
                )


                break


        if chosen is None:

            raise RuntimeError(
                "Balanced nested order failed."
            )


        N,flag = (
            chosen
        )


        j = groups[
            (N,flag)
        ][
            ptr[
                (N,flag)
            ]
        ]


        ptr[
            (N,flag)
        ] += 1


        usedN[
            N
        ] += 1


        used1 += (
            flag
        )


        order.append(
            j
        )


    return np.asarray(
        order,
        dtype=int
    )


# =====================================================================================
# 14. RESUMABLE EXACT TARGETS
# =====================================================================================

def _exact_one(
    j,
    x
):

    return (

        j,

        Rec(
            *x,
            exact_p(
                *x
            )
        )

    )


def exact_set_resumable(
    configs,
    name
):

    full_file = (
        EXACT_CACHE
        /
        f"{name}_full.pkl"
    )


    if full_file.exists():

        ans = safe_pickle_load(
            full_file
        )


        if (
            ans is not None
            and
            len(ans)==len(configs)
        ):

            print(
                f"{name}: full cache loaded "
                f"({len(ans):,})"
            )

            return ans


    folder = (
        EXACT_CACHE
        /
        name
    )


    folder.mkdir(
        parents=True,
        exist_ok=True
    )


    ans = []


    for start in range(
        0,
        len(configs),
        EXACT_CHUNK
    ):

        end = min(
            start+EXACT_CHUNK,
            len(configs)
        )


        file = (

            folder

            /

            f"chunk_{start:05d}_{end:05d}.pkl"

        )


        part = None


        if file.exists():

            part = safe_pickle_load(
                file
            )


            if (
                part is not None
                and
                len(part)==end-start
            ):

                print(
                    f"{name} "
                    f"{start:5d}:{end:5d} | "
                    "Drive cache"
                )


            else:

                part = None


        if part is None:

            jobs = list(
                enumerate(
                    configs[
                        start:end
                    ]
                )
            )


            jobs.sort(
                key=lambda z:
                z[1][3],
                reverse=True
            )


            t0 = (
                time.perf_counter()
            )


            z = Parallel(

                n_jobs=N_EXACT,

                backend="threading"

            )(

                delayed(
                    _exact_one
                )(
                    j,
                    x
                )

                for j,x in jobs

            )


            z.sort(
                key=lambda z:
                z[0]
            )


            part = [
                r
                for _,r in z
            ]


            atomic_pickle(
                part,
                file
            )


            print(
                f"{name} "
                f"{start:5d}:{end:5d} | "
                f"{time.perf_counter()-t0:.1f}s | saved"
            )


        ans.extend(
            part
        )


    atomic_pickle(
        ans,
        full_file
    )


    print(
        f"{name}: COMPLETE."
    )


    return ans


# =====================================================================================
# 15. DATASETS
# =====================================================================================

print(
    "\nGenerating deterministic designs..."
)


train_design = design(
    RMAX,
    cfg.trainN,
    cfg.seed+1
)


val_design = design(
    N_VAL,
    cfg.trainN,
    cfg.seed+2
)


test_seen_design = design(
    N_TEST_SEEN,
    SEEN_N,
    cfg.seed+3
)


test_interp_design = design(
    N_TEST_INTERP,
    INTERP_N,
    cfg.seed+4
)


print(
    "\nGenerating/loading exact targets..."
)


train_raw = exact_set_resumable(
    train_design,
    "TRAIN"
)


val = exact_set_resumable(
    val_design,
    "VAL"
)


test_seen = exact_set_resumable(
    test_seen_design,
    "TEST_SEEN"
)


test_interp = exact_set_resumable(
    test_interp_design,
    "TEST_INTERP"
)


# -------------------------------------------------------------------------
# Balanced nested training sequence
# -------------------------------------------------------------------------

ORDER_FILE = (
    CACHE
    /
    "balanced_training_order.pkl"
)


if ORDER_FILE.exists():

    order = safe_pickle_load(
        ORDER_FILE
    )

else:

    order = balanced_order(
        train_raw,
        cfg.seed+500
    )


    atomic_pickle(
        order,
        ORDER_FILE
    )


train = [

    train_raw[
        int(j)
    ]

    for j in order

]


test_all = (
    test_seen
    +
    test_interp
)


test_split = np.asarray(

    ["seen"]*len(test_seen)

    +

    ["interpolation"]*len(test_interp)

)


print(
    "\nBalanced nested training prefixes:"
)


for R,m in ALLOC:

    subset = (
        train[:R]
    )


    counts = {

        N:
        sum(
            r.N==N
            for r in subset
        )

        for N in cfg.trainN

    }


    print(
        f"R={R:5d}, m={m:3d} | "
        f"N range={min(counts.values())}-{max(counts.values())} | "
        f"i0=1={sum(r.i0==1 for r in subset)/R:.1%}"
    )


# =====================================================================================
# 16. MONTE CARLO SIMULATOR
#
# Exact early termination after C>N.
# =====================================================================================

@njit
def sim_C(
    b,
    g,
    w,
    N,
    i0
):

    S = (
        N-i0
    )

    I = (
        i0
    )

    R = 0
    C = 0


    while I>0:

        inf = (
            b*S*I/N
        )

        rec = (
            g*I
        )

        wan = (
            w*R
        )


        z = np.random.random()*(
            inf+rec+wan
        )


        if z<inf:

            S -= 1
            I += 1
            C += 1


            # Truncated-with-overflow target is now known exactly.
            if C>=N+1:

                return N+1


        elif z<inf+rec:

            I -= 1
            R += 1


        else:

            R -= 1
            S += 1


    return C


# =====================================================================================
# 17. NESTED MC
# =====================================================================================

MC_LEVELS = np.asarray(

    sorted(
        {
            m
            for R,m in ALLOC
        }
    ),

    dtype=np.int64

)


LEVEL_INDEX = {

    int(m):j

    for j,m
    in enumerate(
        MC_LEVELS
    )

}


MAXM = np.zeros(
    RMAX,
    dtype=np.int64
)


for R,m in ALLOC:

    MAXM[:R] = np.maximum(
        MAXM[:R],
        m
    )


ACTUAL_MC_PER_REP = int(
    MAXM.sum()
)


@njit(parallel=True)
def nested_mc_chunk(
    B,
    G,
    W,
    N,
    I0,
    maxm,
    levels,
    maxN,
    global_start,
    seed
):

    n = len(N)

    nlev = len(
        levels
    )


    snap = np.zeros(

        (
            nlev,
            n,
            maxN+2
        ),

        dtype=np.int16

    )


    for j in prange(n):

        global_j = (
            global_start+j
        )


        np.random.seed(
            seed
            +
            100003*global_j
        )


        hist = np.zeros(
            maxN+2,
            dtype=np.int16
        )


        lev = 0


        for q in range(
            1,
            maxm[j]+1
        ):

            c = sim_C(

                B[j],
                G[j],
                W[j],
                N[j],
                I0[j]

            )


            hist[c] += 1


            if (
                lev<nlev
                and
                q==levels[lev]
            ):

                snap[
                    lev,
                    j,
                    :
                ] = hist


                lev += 1


    return snap


def make_mc_labels_resumable(
    records,
    rep
):

    rep_dir = (
        MC_CACHE
        /
        f"rep_{rep:02d}"
    )


    rep_dir.mkdir(
        parents=True,
        exist_ok=True
    )


    final_file = (
        rep_dir
        /
        "labels_final.pkl"
    )


    if final_file.exists():

        ans = safe_pickle_load(
            final_file
        )


        if ans is not None:

            print(
                f"MC rep {rep}: final labels loaded."
            )

            return ans


    B = np.asarray(
        [r.b for r in records],
        dtype=np.float64
    )


    G = np.asarray(
        [r.g for r in records],
        dtype=np.float64
    )


    W = np.asarray(
        [r.w for r in records],
        dtype=np.float64
    )


    N = np.asarray(
        [r.N for r in records],
        dtype=np.int64
    )


    I0 = np.asarray(
        [r.i0 for r in records],
        dtype=np.int64
    )


    mc_seed = (
        cfg.seed
        +
        900000
        +
        rep*1000003
    )


    print(
        f"\nMC replication {rep}/{N_REP}"
    )

    print(
        "Nominal trajectories physically simulated:",
        f"{ACTUAL_MC_PER_REP:,}"
    )


    # compile once
    _ = nested_mc_chunk(

        np.asarray([.8]),
        np.asarray([.5]),
        np.asarray([.1]),

        np.asarray(
            [40],
            dtype=np.int64
        ),

        np.asarray(
            [1],
            dtype=np.int64
        ),

        np.asarray(
            [10],
            dtype=np.int64
        ),

        MC_LEVELS,

        40,
        0,
        mc_seed

    )


    pieces = []


    for start in range(
        0,
        RMAX,
        MC_CHUNK
    ):

        end = min(
            start+MC_CHUNK,
            RMAX
        )


        file = (

            rep_dir

            /

            f"chunk_{start:05d}_{end:05d}.pkl"

        )


        part = None


        if file.exists():

            part = safe_pickle_load(
                file
            )


            if (
                part is not None
                and
                part.shape[1]==end-start
            ):

                print(
                    f"MC rep={rep} "
                    f"{start:5d}:{end:5d} | cache"
                )

            else:

                part = None


        if part is None:

            t0 = (
                time.perf_counter()
            )


            part = nested_mc_chunk(

                B[start:end],
                G[start:end],
                W[start:end],
                N[start:end],
                I0[start:end],

                MAXM[start:end],

                MC_LEVELS,

                max(
                    cfg.trainN
                ),

                start,

                mc_seed

            )


            atomic_pickle(
                part,
                file
            )


            print(
                f"MC rep={rep} "
                f"{start:5d}:{end:5d} | "
                f"{time.perf_counter()-t0:.1f}s | saved"
            )


        pieces.append(
            part
        )


    snap = np.concatenate(
        pieces,
        axis=1
    )


    labels = {}


    for R,m in ALLOC:

        q = (
            LEVEL_INDEX[
                m
            ]
        )


        labels[
            (R,m)
        ] = [

            snap[
                q,
                j,
                :records[j].N+2
            ].astype(
                np.float32
            )
            /
            m

            for j in range(R)

        ]


    atomic_pickle(
        labels,
        final_file
    )


    return labels


# =====================================================================================
# 18. NETWORK
# =====================================================================================

class HazardNet(
    nn.Module
):

    def __init__(
        self
    ):

        super().__init__()


        L = []

        d = 6


        for _ in range(
            cfg.depth
        ):

            L += [

                nn.Linear(
                    d,
                    cfg.width
                ),

                nn.SiLU()

            ]


            d = (
                cfg.width
            )


        L.append(
            nn.Linear(
                d,
                1
            )
        )


        self.net = nn.Sequential(
            *L
        )


    def forward(
        self,
        x
    ):

        return torch.sigmoid(

            self.net(
                x
            ).squeeze(-1)

        )


# =====================================================================================
# 19. PACK DATA BY N
# =====================================================================================

def pack(
    records,
    labels=None
):

    groups = {}


    for j,r in enumerate(
        records
    ):

        groups.setdefault(
            r.N,
            []
        ).append(
            j
        )


    P = {}


    for N,idx in groups.items():

        idx = np.asarray(
            idx,
            dtype=int
        )


        B = len(idx)
        K = N+1


        b = torch.tensor(
            [records[j].b for j in idx],
            dtype=torch.float32
        )[:,None]


        g = torch.tensor(
            [records[j].g for j in idx],
            dtype=torch.float32
        )[:,None]


        w = torch.tensor(
            [records[j].w for j in idx],
            dtype=torch.float32
        )[:,None]


        ns = torch.full(
            (B,1),
            N/Nscale,
            dtype=torch.float32
        )


        i0 = torch.tensor(
            [
                records[j].i0/N
                for j in idx
            ],
            dtype=torch.float32
        )[:,None]


        c = (

            torch.arange(
                K,
                dtype=torch.float32
            )

            /

            N

        )[None,:]


        X = torch.stack(

            [

                b.expand(B,K),
                g.expand(B,K),
                w.expand(B,K),
                ns.expand(B,K),
                i0.expand(B,K),
                c.expand(B,K)

            ],

            dim=2

        ).contiguous()


        Y = np.stack([

            records[j].p

            if labels is None

            else labels[j]

            for j in idx

        ]).astype(
            np.float32
        )


        P[N] = {

            "X":
                X,

            "Y":
                torch.from_numpy(
                    Y
                ),

            "idx":
                idx,

            "n":
                B

        }


    return P


# =====================================================================================
# 20. PMF + TAIL
# =====================================================================================

def pmf_from_h(
    h
):

    B = (
        h.shape[0]
    )


    surv = torch.cat(

        [

            torch.ones(
                (B,1),
                dtype=h.dtype
            ),

            torch.cumprod(
                1-h[:,:-1],
                dim=1
            )

        ],

        dim=1

    )


    return torch.cat(

        [

            surv*h,

            torch.prod(
                1-h,
                dim=1,
                keepdim=True
            )

        ],

        dim=1

    )


def tail(
    P
):

    return torch.flip(

        torch.cumsum(

            torch.flip(
                P[:,1:],
                dims=[1]
            ),

            dim=1

        ),

        dims=[1]

    )


def tail_np(
    p
):

    return np.flip(

        np.cumsum(

            np.flip(
                p[1:]
            )

        )

    )


def predict_batch(
    net,
    X
):

    B,K,_ = (
        X.shape
    )


    h = net(

        X.reshape(
            B*K,
            6
        )

    ).reshape(
        B,
        K
    )


    return pmf_from_h(
        h
    )


# =====================================================================================
# 21. LOSS
# =====================================================================================

def loss_batch(
    net,
    X,
    Y
):

    P = predict_batch(
        net,
        X
    )


    Lp = torch.sum(

        (
            P-Y
        )**2,

        dim=1

    )


    Lrho = torch.mean(

        (

            tail(P)
            -
            tail(Y)

        )**2,

        dim=1

    )


    return (

        Lp
        +
        Lrho

    ).mean()


# =====================================================================================
# 22. BATCHES / VALIDATION
# =====================================================================================

def batches(
    P,
    rng,
    shuffle=True
):

    jobs = []


    for N,G in P.items():

        idx = np.arange(
            G["n"]
        )


        if shuffle:

            rng.shuffle(
                idx
            )


        for s in range(
            0,
            len(idx),
            cfg.batch
        ):

            jobs.append(

                (

                    N,

                    idx[
                        s:
                        s+cfg.batch
                    ]

                )

            )


    if shuffle:

        rng.shuffle(
            jobs
        )


    return jobs


@torch.no_grad()
def val_loss(
    net,
    P
):

    net.eval()


    total = 0.
    n = 0


    rng = np.random.default_rng(
        1
    )


    for N,idx in batches(
        P,
        rng,
        False
    ):

        L = loss_batch(

            net,

            P[N]["X"][idx],

            P[N]["Y"][idx]

        )


        total += (
            L.item()
            *
            len(idx)
        )


        n += (
            len(idx)
        )


    return (
        total/n
    )


# =====================================================================================
# 23. RESUMABLE TRAINING
# =====================================================================================

def fit_resumable(
    TR,
    VA,
    seed,
    name
):

    final_file = (

        MODEL_CACHE

        /

        f"{name}_FINAL.pt"

    )


    ckpt_file = (

        CHECKPOINT_CACHE

        /

        f"{name}_CHECKPOINT.pt"

    )


    # -------------------------------------------------------------------------
    # Completed
    # -------------------------------------------------------------------------

    if final_file.exists():

        x = torch.load(

            final_file,

            map_location="cpu",

            weights_only=False

        )


        net = HazardNet()


        net.load_state_dict(
            x[
                "state"
            ]
        )


        return (

            net,

            float(
                x[
                    "training_sec"
                ]
            ),

            x

        )


    # -------------------------------------------------------------------------
    # New
    # -------------------------------------------------------------------------

    seed_all(
        seed
    )


    net = HazardNet()


    optimizer = torch.optim.AdamW(

        net.parameters(),

        lr=cfg.lr,

        weight_decay=cfg.wd

    )


    scheduler = (
        torch.optim.lr_scheduler.ReduceLROnPlateau(

            optimizer,

            factor=.5,

            patience=15

        )
    )


    rng = np.random.default_rng(
        seed+77
    )


    start_epoch = 1

    best = np.inf
    best_epoch = 0
    best_state = None
    wait = 0

    history = []

    previous_elapsed = 0.


    # -------------------------------------------------------------------------
    # Resume
    # -------------------------------------------------------------------------

    if ckpt_file.exists():

        ck = torch.load(

            ckpt_file,

            map_location="cpu",

            weights_only=False

        )


        net.load_state_dict(
            ck[
                "model"
            ]
        )


        optimizer.load_state_dict(
            ck[
                "optimizer"
            ]
        )


        scheduler.load_state_dict(
            ck[
                "scheduler"
            ]
        )


        start_epoch = (
            ck[
                "epoch"
            ]
            +
            1
        )


        best = (
            ck[
                "best"
            ]
        )


        best_epoch = (
            ck[
                "best_epoch"
            ]
        )


        best_state = (
            ck[
                "best_state"
            ]
        )


        wait = (
            ck[
                "wait"
            ]
        )


        history = list(
            ck.get(
                "history",
                []
            )
        )


        previous_elapsed = float(
            ck.get(
                "elapsed_sec",
                0.
            )
        )


        rng.bit_generator.state = (
            ck[
                "rng_state"
            ]
        )


        if "torch_rng" in ck:

            torch.set_rng_state(
                ck[
                    "torch_rng"
                ]
            )


        print(
            f"{name}: RESUME epoch {start_epoch}"
        )


    session_start = (
        time.perf_counter()
    )


    stopped_epoch = (
        cfg.epochs
    )


    for epoch in range(
        start_epoch,
        cfg.epochs+1
    ):

        net.train()


        for N,idx in batches(
            TR,
            rng,
            True
        ):

            optimizer.zero_grad(
                set_to_none=True
            )


            L = loss_batch(

                net,

                TR[N]["X"][idx],

                TR[N]["Y"][idx]

            )


            if not torch.isfinite(
                L
            ):

                raise RuntimeError(
                    f"{name}: non-finite loss."
                )


            L.backward()


            torch.nn.utils.clip_grad_norm_(

                net.parameters(),

                cfg.clip

            )


            optimizer.step()


        v = val_loss(
            net,
            VA
        )


        scheduler.step(
            v
        )


        history.append(
            float(v)
        )


        if (
            best_state is None
            or
            v<best-cfg.delta
        ):

            best = float(
                v
            )

            best_epoch = (
                epoch
            )

            wait = 0


            best_state = {

                k:
                x.detach().clone()

                for k,x
                in net.state_dict().items()

            }


        else:

            wait += 1


        if (
            epoch==1
            or
            epoch%CHECKPOINT_EVERY==0
        ):

            elapsed = (

                previous_elapsed

                +

                (
                    time.perf_counter()
                    -
                    session_start
                )

            )


            atomic_torch_save(

                {

                    "epoch":
                        epoch,

                    "model":
                        net.state_dict(),

                    "optimizer":
                        optimizer.state_dict(),

                    "scheduler":
                        scheduler.state_dict(),

                    "best":
                        best,

                    "best_epoch":
                        best_epoch,

                    "best_state":
                        best_state,

                    "wait":
                        wait,

                    "history":
                        history,

                    "rng_state":
                        rng.bit_generator.state,

                    "torch_rng":
                        torch.get_rng_state(),

                    "elapsed_sec":
                        elapsed

                },

                ckpt_file

            )


            print(
                f"{name} | "
                f"ep={epoch:3d} | "
                f"val={v:.4e} | "
                f"best={best:.4e} | "
                f"wait={wait}"
            )


        if wait>=cfg.patience:

            stopped_epoch = (
                epoch
            )

            print(
                f"{name}: early stop at {epoch}"
            )

            break


    elapsed = (

        previous_elapsed

        +

        (
            time.perf_counter()
            -
            session_start
        )

    )


    net.load_state_dict(
        best_state
    )


    final_payload = {

        "state":
            best_state,

        "best_epoch":
            best_epoch,

        "stopped_epoch":
            stopped_epoch,

        "validation":
            best,

        "history":
            history,

        "training_sec":
            elapsed

    }


    atomic_torch_save(
        final_payload,
        final_file
    )


    if ckpt_file.exists():

        ckpt_file.unlink()


    return (
        net,
        elapsed,
        final_payload
    )


# =====================================================================================
# 24. TEST ERRORS IN ORIGINAL TEST-SET ORDER
# =====================================================================================

@torch.no_grad()
def metric_arrays(
    net,
    P,
    n_total
):

    net.eval()


    E2 = np.full(
        n_total,
        np.nan
    )


    Erho = np.full(
        n_total,
        np.nan
    )


    for N,G in P.items():

        for s in range(
            0,
            G["n"],
            cfg.batch
        ):

            e = min(
                s+cfg.batch,
                G["n"]
            )


            X = G["X"][
                s:e
            ]


            Y = G["Y"][
                s:e
            ]


            idx = G["idx"][
                s:e
            ]


            Ph = predict_batch(
                net,
                X
            )


            E2[idx] = (

                torch.linalg.vector_norm(
                    Ph-Y,
                    dim=1
                )

                .cpu()
                .numpy()

            )


            Erho[idx] = (

                torch.max(

                    torch.abs(

                        tail(Ph)
                        -
                        tail(Y)

                    ),

                    dim=1

                ).values

                .cpu()
                .numpy()

            )


    return (
        E2,
        Erho
    )


# =====================================================================================
# 25. MC TEACHER-LABEL ERROR
# =====================================================================================

def label_errors(
    records,
    labels
):

    E2 = []
    Erho = []


    for r,p_mc in zip(
        records,
        labels
    ):

        E2.append(

            np.linalg.norm(
                np.asarray(
                    p_mc
                )
                -
                r.p
            )

        )


        Erho.append(

            np.max(

                np.abs(

                    tail_np(
                        np.asarray(
                            p_mc
                        )
                    )

                    -

                    tail_np(
                        r.p
                    )

                )

            )

        )


    return (

        np.asarray(
            E2,
            dtype=float
        ),

        np.asarray(
            Erho,
            dtype=float
        )

    )


# =====================================================================================
# 26. PACK COMMON VALIDATION / TEST
# =====================================================================================

VA = pack(
    val
)


TE = pack(
    test_all
)


# =====================================================================================
# 27. PERSISTENT EXPERIMENT PROGRESS
# =====================================================================================

PROGRESS_FILE = (
    ROOT
    /
    "experiment_progress.pkl"
)


progress = safe_pickle_load(

    PROGRESS_FILE,

    default=None

)


if progress is None:

    progress = {

        "exact":
            {},

        "mc":
            {},

        "label_errors":
            {}

    }


# =====================================================================================
# 28. REPLICATION LOOP
# =====================================================================================

for rep in range(
    1,
    N_REP+1
):

    print(
        "\n"
        +
        "="*100
    )

    print(
        f"REPLICATION {rep}/{N_REP}"
    )

    print(
        "="*100
    )


    train_seed = (
        cfg.seed
        +
        7001
        +
        rep*100003
    )


    # =========================================================================
    # A. EXACT-TEACHER REFERENCE
    #
    # This is an oracle-style benchmark, not a fixed-MC-budget competitor.
    # =========================================================================

    exact_key = (
        f"rep{rep:02d}"
    )


    if exact_key not in progress[
        "exact"
    ]:

        print(
            "\nEXACT TEACHER | R=5000"
        )


        TR_exact = pack(
            train
        )


        exact_net,training_sec,fitmeta = fit_resumable(

            TR_exact,

            VA,

            train_seed,

            f"rep{rep:02d}_EXACT_R5000"

        )


        e2,erho = metric_arrays(

            exact_net,

            TE,

            len(
                test_all
            )

        )


        progress[
            "exact"
        ][exact_key] = {

            "E2":
                e2,

            "Erho":
                erho,

            "training_sec":
                training_sec,

            "best_epoch":
                fitmeta[
                    "best_epoch"
                ],

            "stopped_epoch":
                fitmeta[
                    "stopped_epoch"
                ]

        }


        atomic_pickle(
            progress,
            PROGRESS_FILE
        )


        del (
            TR_exact,
            exact_net
        )


    # =========================================================================
    # B. MONTE CARLO LABELS
    # =========================================================================

    labels_all = make_mc_labels_resumable(
        train,
        rep
    )


    # =========================================================================
    # C. LABEL-NOISE DIAGNOSTICS
    # =========================================================================

    for R,m in ALLOC:

        key = (
            f"rep{rep:02d}_R{R}_m{m}"
        )


        if key not in progress[
            "label_errors"
        ]:

            le2,lerho = label_errors(

                train[:R],

                labels_all[
                    (R,m)
                ]

            )


            progress[
                "label_errors"
            ][key] = {

                "rep":
                    rep,

                "R":
                    R,

                "m":
                    m,

                "E2":
                    le2,

                "Erho":
                    lerho

            }


            atomic_pickle(
                progress,
                PROGRESS_FILE
            )


    # =========================================================================
    # D. MC-TRAINED EMULATORS
    # =========================================================================

    for R,m in ALLOC:

        key = (
            f"rep{rep:02d}_R{R}_m{m}"
        )


        if key in progress[
            "mc"
        ]:

            print(
                f"{key}: complete — skipping."
            )

            continue


        print(
            "\n"
            +
            "-"*90
        )


        print(
            f"MC TEACHER | rep={rep} | "
            f"R={R:,} | m={m} | Rm={R*m:,}"
        )


        print(
            "-"*90
        )


        TR = pack(

            train[:R],

            labels_all[
                (R,m)
            ]

        )


        net,training_sec,fitmeta = fit_resumable(

            TR,

            VA,

            train_seed,

            f"rep{rep:02d}_MC_R{R}_m{m}"

        )


        e2,erho = metric_arrays(

            net,

            TE,

            len(
                test_all
            )

        )


        progress[
            "mc"
        ][key] = {

            "rep":
                rep,

            "R":
                R,

            "m":
                m,

            "budget":
                R*m,

            "E2":
                e2,

            "Erho":
                erho,

            "training_sec":
                training_sec,

            "best_epoch":
                fitmeta[
                    "best_epoch"
                ],

            "stopped_epoch":
                fitmeta[
                    "stopped_epoch"
                ]

        }


        atomic_pickle(
            progress,
            PROGRESS_FILE
        )


        print(
            f"E2 median={np.median(e2):.6g} | "
            f"E_rho median={np.median(erho):.6g}"
        )


        del (
            TR,
            net
        )


# =====================================================================================
# 29. BUILD ERROR MATRICES
# =====================================================================================

def exact_matrix(
    metric
):

    return np.vstack([

        progress[
            "exact"
        ][
            f"rep{rep:02d}"
        ][metric]

        for rep in range(
            1,
            N_REP+1
        )

    ])


def mc_matrix(
    R,
    m,
    metric
):

    return np.vstack([

        progress[
            "mc"
        ][
            f"rep{rep:02d}_R{R}_m{m}"
        ][metric]

        for rep in range(
            1,
            N_REP+1
        )

    ])


def label_matrix(
    R,
    m,
    metric
):

    return np.vstack([

        progress[
            "label_errors"
        ][
            f"rep{rep:02d}_R{R}_m{m}"
        ][metric]

        for rep in range(
            1,
            N_REP+1
        )

    ])


# =====================================================================================
# 30. HIERARCHICAL PAIRED BOOTSTRAP
#
# Resampling units:
#   level 1: independent MC/training replication
#   level 2: test configuration
#
# The same resampled indices can be used across allocations when contrasts are formed.
# =====================================================================================

def hierarchical_ci(
    A,
    B=BOOTSTRAP_B,
    seed=12345,
    q=.50
):

    A = np.asarray(
        A,
        dtype=float
    )


    nr,nc = (
        A.shape
    )


    rng = np.random.default_rng(
        seed
    )


    vals = np.empty(
        B
    )


    for b in range(B):

        ir = rng.integers(
            0,
            nr,
            size=nr
        )


        ic = rng.integers(
            0,
            nc,
            size=nc
        )


        vals[b] = np.quantile(

            A[
                ir
            ][
                :,
                ic
            ],

            q

        )


    return (

        float(
            np.quantile(
                vals,
                .025
            )
        ),

        float(
            np.quantile(
                vals,
                .975
            )
        )

    )


def hierarchical_contrast_ci(
    A,
    Bmat,
    B=BOOTSTRAP_B,
    seed=12345
):

    A = np.asarray(
        A,
        dtype=float
    )


    Bmat = np.asarray(
        Bmat,
        dtype=float
    )


    assert (
        A.shape
        ==
        Bmat.shape
    )


    nr,nc = (
        A.shape
    )


    rng = np.random.default_rng(
        seed
    )


    delta = np.empty(
        B
    )


    for b in range(B):

        ir = rng.integers(
            0,
            nr,
            size=nr
        )


        ic = rng.integers(
            0,
            nc,
            size=nc
        )


        As = A[
            ir
        ][
            :,
            ic
        ]


        Bs = Bmat[
            ir
        ][
            :,
            ic
        ]


        delta[b] = (

            np.median(
                As
            )

            -

            np.median(
                Bs
            )

        )


    return (

        float(
            np.median(A)
            -
            np.median(Bmat)
        ),

        float(
            np.quantile(
                delta,
                .025
            )
        ),

        float(
            np.quantile(
                delta,
                .975
            )
        )

    )


# =====================================================================================
# 31. PRIMARY NUMERICAL SUMMARIES
# =====================================================================================

def matrix_summary(
    A,
    seed
):

    x = np.asarray(
        A,
        dtype=float
    )


    med = float(
        np.median(
            x
        )
    )


    q1 = float(
        np.quantile(
            x,
            .25
        )
    )


    q3 = float(
        np.quantile(
            x,
            .75
        )
    )


    p95 = float(
        np.quantile(
            x,
            .95
        )
    )


    lo,hi = hierarchical_ci(
        x,
        seed=seed
    )


    return {

        "median":
            med,

        "q1":
            q1,

        "q3":
            q3,

        "p95":
            p95,

        "CI_low":
            lo,

        "CI_high":
            hi

    }


summary_rows = []


# -------------------------------------------------------------------------
# Exact reference
# -------------------------------------------------------------------------

exact_E2 = exact_matrix(
    "E2"
)

exact_Erho = exact_matrix(
    "Erho"
)


sE2 = matrix_summary(
    exact_E2,
    cfg.seed+101
)


sEr = matrix_summary(
    exact_Erho,
    cfg.seed+102
)


summary_rows.append({

    "Teacher":
        "Exact reference",

    "R":
        RMAX,

    "m":
        np.nan,

    "budget":
        np.nan,

    "E2_med":
        sE2["median"],

    "E2_q1":
        sE2["q1"],

    "E2_q3":
        sE2["q3"],

    "E2_p95":
        sE2["p95"],

    "E2_CI_low":
        sE2["CI_low"],

    "E2_CI_high":
        sE2["CI_high"],

    "Erho_med":
        sEr["median"],

    "Erho_q1":
        sEr["q1"],

    "Erho_q3":
        sEr["q3"],

    "Erho_p95":
        sEr["p95"],

    "Erho_CI_low":
        sEr["CI_low"],

    "Erho_CI_high":
        sEr["CI_high"],

    "label_E2_med":
        0.,

    "label_Erho_med":
        0.

})


# -------------------------------------------------------------------------
# MC allocations
# -------------------------------------------------------------------------

for k,(R,m) in enumerate(
    ALLOC
):

    A2 = mc_matrix(
        R,
        m,
        "E2"
    )


    Ar = mc_matrix(
        R,
        m,
        "Erho"
    )


    L2 = label_matrix(
        R,
        m,
        "E2"
    )


    Lr = label_matrix(
        R,
        m,
        "Erho"
    )


    s2 = matrix_summary(
        A2,
        cfg.seed+1000+k
    )


    sr = matrix_summary(
        Ar,
        cfg.seed+2000+k
    )


    summary_rows.append({

        "Teacher":
            "Monte Carlo",

        "R":
            R,

        "m":
            m,

        "budget":
            R*m,

        "E2_med":
            s2["median"],

        "E2_q1":
            s2["q1"],

        "E2_q3":
            s2["q3"],

        "E2_p95":
            s2["p95"],

        "E2_CI_low":
            s2["CI_low"],

        "E2_CI_high":
            s2["CI_high"],

        "Erho_med":
            sr["median"],

        "Erho_q1":
            sr["q1"],

        "Erho_q3":
            sr["q3"],

        "Erho_p95":
            sr["p95"],

        "Erho_CI_low":
            sr["CI_low"],

        "Erho_CI_high":
            sr["CI_high"],

        "label_E2_med":
            float(
                np.median(
                    L2
                )
            ),

        "label_Erho_med":
            float(
                np.median(
                    Lr
                )
            )

    })


summary = pd.DataFrame(
    summary_rows
)


summary.to_csv(

    OUT
    /
    "main_table_5_3B_numeric.csv",

    index=False

)


# =====================================================================================
# 32. PUBLICATION-FORMATTED MAIN TABLE
# =====================================================================================

def fmt_ci(
    med,
    lo,
    hi
):

    return (
        f"{med:.4g} "
        f"[{lo:.4g}, {hi:.4g}]"
    )


pretty_rows = []


for _,z in summary.iterrows():

    pretty_rows.append({

        "Teacher":
            z["Teacher"],

        "R":
            int(
                z["R"]
            ),

        "m":
            (
                "--"
                if pd.isna(
                    z["m"]
                )
                else
                str(
                    int(
                        z["m"]
                    )
                )
            ),

        "Rm":
            (
                "--"
                if pd.isna(
                    z["budget"]
                )
                else
                f"{int(z['budget']):,}"
            ),

        "E2 median [95% CI]":
            fmt_ci(
                z["E2_med"],
                z["E2_CI_low"],
                z["E2_CI_high"]
            ),

        "E2 p95":
            f"{z['E2_p95']:.4g}",

        "Erho median [95% CI]":
            fmt_ci(
                z["Erho_med"],
                z["Erho_CI_low"],
                z["Erho_CI_high"]
            ),

        "Erho p95":
            f"{z['Erho_p95']:.4g}",

        "MC-label E2":
            f"{z['label_E2_med']:.4g}",

        "MC-label Erho":
            f"{z['label_Erho_med']:.4g}"

    })


pretty = pd.DataFrame(
    pretty_rows
)


pretty.to_csv(

    OUT
    /
    "main_table_5_3B.csv",

    index=False

)


(
    OUT
    /
    "main_table_5_3B.tex"
).write_text(

    pretty.to_latex(
        index=False,
        escape=False
    )

)


print(
    "\n"
    +
    "="*130
)

print(
    "MAIN TABLE — 5.3-B"
)

print(
    "="*130
)


print(
    pretty.to_string(
        index=False
    )
)


# =====================================================================================
# 33. RAW TEST-ERROR DATAFRAME
# =====================================================================================

raw_rows = []


for rep in range(
    1,
    N_REP+1
):

    for R,m in ALLOC:

        key = (
            f"rep{rep:02d}_R{R}_m{m}"
        )


        e2 = progress[
            "mc"
        ][key][
            "E2"
        ]


        er = progress[
            "mc"
        ][key][
            "Erho"
        ]


        for j,r in enumerate(
            test_all
        ):

            raw_rows.append({

                "rep":
                    rep,

                "R":
                    R,

                "m":
                    m,

                "budget":
                    R*m,

                "split":
                    test_split[j],

                "index":
                    j,

                "N":
                    r.N,

                "i0":
                    r.i0,

                "R0":
                    r.b/r.g,

                "E2":
                    e2[j],

                "Erho":
                    er[j]

            })


raw = pd.DataFrame(
    raw_rows
)


raw.to_csv(

    OUT
    /
    "raw_test_errors.csv",

    index=False

)


# =====================================================================================
# 34. RAW TEACHER-LABEL ERRORS
# =====================================================================================

label_rows = []


for rep in range(
    1,
    N_REP+1
):

    for R,m in ALLOC:

        key = (
            f"rep{rep:02d}_R{R}_m{m}"
        )


        z = progress[
            "label_errors"
        ][key]


        for j in range(R):

            label_rows.append({

                "rep":
                    rep,

                "R":
                    R,

                "m":
                    m,

                "index":
                    j,

                "N":
                    train[j].N,

                "i0":
                    train[j].i0,

                "R0":
                    train[j].b/train[j].g,

                "E2":
                    z["E2"][j],

                "Erho":
                    z["Erho"][j]

            })


label_df = pd.DataFrame(
    label_rows
)


label_df.to_csv(

    OUT
    /
    "teacher_label_errors.csv",

    index=False

)


# =====================================================================================
# 35. ROBUSTNESS BY SCIENTIFIC STRATUM
#
# These are intended primarily for supplement / reviewer checks.
# =====================================================================================

raw[
    "R0_regime"
] = pd.cut(

    raw["R0"],

    bins=[
        -np.inf,
        1.,
        2.,
        np.inf
    ],

    labels=[
        r"$R_0<1$",
        r"$1\leq R_0<2$",
        r"$R_0\geq2$"
    ]

)


raw[
    "i0_group"
] = np.where(

    raw[
        "i0"
    ]==1,

    r"$i_0=1$",

    r"$i_0>1$"

)


robust_rows = []


for stratum_type,column in (

    (
        "population-grid",
        "split"
    ),

    (
        "R0-regime",
        "R0_regime"
    ),

    (
        "initial-infections",
        "i0_group"
    )

):

    for (
        R,
        m,
        level
    ),z in raw.groupby(
        [
            "R",
            "m",
            column
        ],
        observed=True
    ):

        robust_rows.append({

            "stratum_type":
                stratum_type,

            "stratum":
                str(level),

            "R":
                int(R),

            "m":
                int(m),

            "n":
                len(z),

            "median_E2":
                float(
                    np.median(
                        z["E2"]
                    )
                ),

            "p95_E2":
                float(
                    np.quantile(
                        z["E2"],
                        .95
                    )
                ),

            "median_Erho":
                float(
                    np.median(
                        z["Erho"]
                    )
                ),

            "p95_Erho":
                float(
                    np.quantile(
                        z["Erho"],
                        .95
                    )
                )

        })


robust = pd.DataFrame(
    robust_rows
)


robust.to_csv(

    OUT
    /
    "robustness_by_stratum.csv",

    index=False

)


# =====================================================================================
# 36. CORE EXTREME-ALLOCATION CONTRAST
#
# Precision-rich:
#   R=200, m=250
#
# Coverage-rich:
#   R=5000, m=10
#
# Difference is:
#
#   median error(coverage-rich)
#   -
#   median error(precision-rich)
#
# Negative => advantage for greater configuration coverage.
# =====================================================================================

contrast_rows = []


for metric in (
    "E2",
    "Erho"
):

    A = mc_matrix(
        5000,
        10,
        metric
    )


    Bmat = mc_matrix(
        200,
        250,
        metric
    )


    delta,lo,hi = hierarchical_contrast_ci(

        A,
        Bmat,

        seed=(
            cfg.seed
            +
            (
                7000
                if metric=="E2"
                else
                8000
            )
        )

    )


    contrast_rows.append({

        "metric":
            metric,

        "contrast":
            (
                "coverage-rich minus precision-rich"
            ),

        "estimate":
            delta,

        "CI_low":
            lo,

        "CI_high":
            hi,

        "interpretation":
            (
                "coverage-rich lower error"
                if delta<0
                else
                "precision-rich lower error"
            )

    })


contrast = pd.DataFrame(
    contrast_rows
)


contrast.to_csv(

    OUT
    /
    "extreme_allocation_contrast.csv",

    index=False

)


print(
    "\nEXTREME-ALLOCATION CONTRAST"
)

print(
    contrast.to_string(
        index=False
    )
)


# =====================================================================================
# 37. MAIN JASA-STYLE FIGURE
#
# A. Emulator distributional error
# B. Emulator tail-risk error
# C. MC teacher-label distributional error
# D. MC teacher-label tail-risk error
#
# This figure directly connects:
#
#   simulation allocation
#       ->
#   label noise
#       ->
#   emulator accuracy.
# =====================================================================================

mc_summary = summary[
    summary[
        "Teacher"
    ]=="Monte Carlo"
].copy()


x = mc_summary[
    "R"
].to_numpy(
    dtype=float
)


mvals = mc_summary[
    "m"
].to_numpy(
    dtype=int
)


fig,ax = plt.subplots(

    2,
    2,

    figsize=(
        11.5,
        8.2
    )

)


# -------------------------------------------------------------------------
# A. Emulator E2
# -------------------------------------------------------------------------

y = mc_summary[
    "E2_med"
].to_numpy()


lo = mc_summary[
    "E2_CI_low"
].to_numpy()


hi = mc_summary[
    "E2_CI_high"
].to_numpy()


ax[0,0].errorbar(

    x,
    y,

    yerr=np.vstack(
        [
            y-lo,
            hi-y
        ]
    ),

    fmt="o-",

    color="#0072B2",

    lw=2,

    capsize=3

)


exact_row = summary[
    summary[
        "Teacher"
    ]=="Exact reference"
].iloc[0]


ax[0,0].axhline(

    exact_row[
        "E2_med"
    ],

    color="#D55E00",

    ls="--",

    lw=2,

    label="Exact-teacher reference"

)


ax[0,0].axhspan(

    exact_row[
        "E2_CI_low"
    ],

    exact_row[
        "E2_CI_high"
    ],

    color="#D55E00",

    alpha=.10,

    linewidth=0

)


# -------------------------------------------------------------------------
# B. Emulator E_rho
# -------------------------------------------------------------------------

y2 = mc_summary[
    "Erho_med"
].to_numpy()


lo2 = mc_summary[
    "Erho_CI_low"
].to_numpy()


hi2 = mc_summary[
    "Erho_CI_high"
].to_numpy()


ax[0,1].errorbar(

    x,
    y2,

    yerr=np.vstack(
        [
            y2-lo2,
            hi2-y2
        ]
    ),

    fmt="o-",

    color="#009E73",

    lw=2,

    capsize=3

)


ax[0,1].axhline(

    exact_row[
        "Erho_med"
    ],

    color="#D55E00",

    ls="--",

    lw=2,

    label="Exact-teacher reference"

)


ax[0,1].axhspan(

    exact_row[
        "Erho_CI_low"
    ],

    exact_row[
        "Erho_CI_high"
    ],

    color="#D55E00",

    alpha=.10,

    linewidth=0

)


# -------------------------------------------------------------------------
# C. Teacher-label E2
# -------------------------------------------------------------------------

label_E2_med = []

label_E2_q1 = []
label_E2_q3 = []


label_Erho_med = []

label_Erho_q1 = []
label_Erho_q3 = []


for R,m in ALLOC:

    z2 = label_matrix(
        R,
        m,
        "E2"
    )


    zr = label_matrix(
        R,
        m,
        "Erho"
    )


    label_E2_med.append(
        np.median(
            z2
        )
    )

    label_E2_q1.append(
        np.quantile(
            z2,
            .25
        )
    )

    label_E2_q3.append(
        np.quantile(
            z2,
            .75
        )
    )


    label_Erho_med.append(
        np.median(
            zr
        )
    )

    label_Erho_q1.append(
        np.quantile(
            zr,
            .25
        )
    )

    label_Erho_q3.append(
        np.quantile(
            zr,
            .75
        )
    )


label_E2_med = np.asarray(
    label_E2_med
)

label_E2_q1 = np.asarray(
    label_E2_q1
)

label_E2_q3 = np.asarray(
    label_E2_q3
)


label_Erho_med = np.asarray(
    label_Erho_med
)

label_Erho_q1 = np.asarray(
    label_Erho_q1
)

label_Erho_q3 = np.asarray(
    label_Erho_q3
)


ax[1,0].plot(

    x,
    label_E2_med,

    "o-",

    color="#CC79A7",

    lw=2

)


ax[1,0].fill_between(

    x,
    label_E2_q1,
    label_E2_q3,

    color="#CC79A7",

    alpha=.15

)


# -------------------------------------------------------------------------
# D. Teacher-label E_rho
# -------------------------------------------------------------------------

ax[1,1].plot(

    x,
    label_Erho_med,

    "o-",

    color="#E69F00",

    lw=2

)


ax[1,1].fill_between(

    x,
    label_Erho_q1,
    label_Erho_q3,

    color="#E69F00",

    alpha=.15

)


# -------------------------------------------------------------------------
# Shared styling
# -------------------------------------------------------------------------

titles = [

    "(A) Emulator distributional error",

    "(B) Emulator tail-risk error",

    "(C) MC teacher-label error",

    "(D) MC teacher tail-risk error"

]


ylabs = [

    r"Median $E_2$",

    r"Median $E_\rho$",

    r"Teacher-label $E_2$",

    r"Teacher-label $E_\rho$"

]


for a,title,ylabel in zip(
    ax.flat,
    titles,
    ylabs
):

    a.set_xscale(
        "log"
    )


    a.set_yscale(
        "log"
    )


    a.set_xticks(
        x
    )


    a.set_xticklabels(
        [
            f"{int(v)}"
            for v in x
        ]
    )


    a.set_xlabel(
        r"Training configurations $R$"
    )


    a.set_ylabel(
        ylabel
    )


    a.set_title(
        title
    )


    a.grid(
        alpha=.15
    )


    for xi,mi in zip(
        x,
        mvals
    ):

        # place m at the bottom of each panel
        ymin,ymax = a.get_ylim()

        a.annotate(

            rf"$m={mi}$",

            xy=(
                xi,
                ymin
            ),

            xytext=(
                0,
                6
            ),

            textcoords="offset points",

            ha="center",

            va="bottom",

            fontsize=8

        )


ax[0,0].legend(
    frameon=False,
    fontsize=9
)


ax[0,1].legend(
    frameon=False,
    fontsize=9
)


fig.suptitle(

    r"Coverage versus Monte Carlo label precision "
    r"under the fixed budget $Rm=50{,}000$",

    fontsize=14

)


plt.tight_layout()


plt.savefig(

    OUT
    /
    "figure_5_3B_main.pdf",

    bbox_inches="tight"

)


plt.savefig(

    OUT
    /
    "figure_5_3B_main.png",

    dpi=300,

    bbox_inches="tight"

)


plt.show()


# =====================================================================================
# 38. FIGURE CAPTION — SAVE WITH RESULTS
# =====================================================================================

caption = r"""
Coverage versus Monte Carlo label precision under a fixed simulation budget
Rm=50,000. Panels (A) and (B) report the median distributional and tail-risk
errors of the neural emulator across independent Monte Carlo/training
replications and exact test configurations; error bars are hierarchical
paired-bootstrap 95% confidence intervals. The dashed line and shaded band
give the corresponding exact-teacher reference. Panels (C) and (D) report
the Monte Carlo teacher-label errors, with shaded interquartile ranges.
Configurations and Monte Carlo trajectories are nested across allocations,
so increasing R decreases m while preserving the nominal simulation budget.
The exact-teacher fit is an oracle-style reference and is not a fixed-cost
competitor.
""".strip()


(
    OUT
    /
    "figure_5_3B_caption.txt"
).write_text(
    caption
)


# =====================================================================================
# 39. FINAL RESULT OBJECT
# =====================================================================================

FINAL_FILE = (
    OUT
    /
    "section_5_3B_final_results.pkl"
)


atomic_pickle(

    {

        "scientific_config":
            SCIENTIFIC_CONFIG,

        "signature":
            CURRENT_SIGNATURE,

        "summary":
            summary,

        "pretty_table":
            pretty,

        "contrast":
            contrast,

        "robustness":
            robust,

        "actual_MC_trajectories_per_rep":
            ACTUAL_MC_PER_REP,

        "nominal_budget_per_allocation":
            BUDGET,

        "N_REP":
            N_REP,

        "validation_protocol":
            SCIENTIFIC_CONFIG[
                "validation_protocol"
            ]

    },

    FINAL_FILE

)


# =====================================================================================
# 40. FINAL STATUS
# =====================================================================================

print(
    "\n"
    +
    "="*110
)

print(
    "EXPERIMENT 5.3-B COMPLETE"
)

print(
    "="*110
)


print(
    "Run directory:"
)

print(
    ROOT
)


print()


print(
    "Independent MC/training replications:",
    N_REP
)


print(
    "Fixed nominal MC budget/allocation:",
    f"{BUDGET:,}"
)


print(
    "Physical nested trajectories/replication:",
    f"{ACTUAL_MC_PER_REP:,}"
)


print(
    "Physical nested trajectories total:",
    f"{ACTUAL_MC_PER_REP*N_REP:,}"
)


print()


print(
    "Exact validation configurations:",
    N_VAL
)


print(
    "Seen-N test configurations:",
    N_TEST_SEEN
)


print(
    "Held-out-N test configurations:",
    N_TEST_INTERP
)


print()


print(
    "Main table:"
)

print(
    OUT
    /
    "main_table_5_3B.tex"
)


print(
    "Main figure:"
)

print(
    OUT
    /
    "figure_5_3B_main.pdf"
)


print(
    "Raw test errors:"
)

print(
    OUT
    /
    "raw_test_errors.csv"
)


print(
    "Teacher-label diagnostics:"
)

print(
    OUT
    /
    "teacher_label_errors.csv"
)


print(
    "Robustness diagnostics:"
)

print(
    OUT
    /
    "robustness_by_stratum.csv"
)


print(
    "Coverage-vs-precision contrast:"
)

print(
    OUT
    /
    "extreme_allocation_contrast.csv"
)


print()


print(
    "AFTER A COLAB DISCONNECT:"
)

print(
    "1. Reconnect."
)

print(
    "2. Run this same cell."
)

print(
    "3. Choose 1 = RESUME."
)

print(
    "4. Completed exact chunks, MC chunks, models, "
    "replications and allocations are skipped."
)

print()


print(
    "IF YOU CHANGE SCIENTIFIC CODE:"
)

print(
    "1. Increment CODE_VERSION."
)

print(
    "2. Rerun."
)

print(
    "3. Choose 2 = START NEW."
)

print(
    "This prevents heterogeneous computations from being mixed."
)


print(
    "="*110
)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).

EXPERIMENT 5.3-B — START MODE
1 = RESUME latest experiment
2 = START NEW experiment from zero
Choose 1 or 2: 1

RESUMING EXISTING EXPERIMENT
ROOT: /content/drive/MyDrive/StatisticalLearning/Experiment_5_3B_JASA/run_20260823_082952
Scientific signature: 8f7b73af5fdd6484

COMPUTATIONAL CONFIGURATION
CPU cores: 2
Exact workers: 2
Numba threads: 2
PyTorch threads: 2
Replications: 5
MC budget/allocation: 50,000
Allocations: ((200, 250), (500, 100), (1000, 50), (2000, 25), (5000, 10))

Generating deterministic designs...

Generating/loading exact targets...
TRAIN     0:  100 | Drive cache
TRAIN   100:  200 | Drive cache
TRAIN   200:  300 | Drive cache
TRAIN   300:  400 | 168.6s | saved
TRAIN   400:  500 | 125.6s | saved
TRAIN   500:  600 | 160.6s | saved
TRAIN   600:  700 | 142.3s | saved
TRAIN   700:  800 | 137.3s | saved
